# **Auditor de Logs y Guardrails para Prompts de IA Generativa**
Nivel: Intermedio
## **OBJETIVO DE NEGOCIO**
Supervisar el funcionamiento de un asistente de IA en producción para garantizar respuestas rápidas,
satisfactorias y libres de fugas de información confidencial.
## QUÉ DEBE HACER TU CÓDIGO (GUÍA DE TRABAJO)
1- Detectar riesgos de seguridad (Guardrails): Revisar el texto de cada respuesta buscando si contiene alguna
de las palabras_prohibidas (sin importar si están en mayúsculas o minúsculas) y marcar esos
registros como alerta de seguridad.

2- Evaluar rendimiento técnico: Identificar qué interacciones superaron el tiempo máximo de respuesta
permitido (3.0 segundos) y calcular el consumo total y promedio de tokens.


3-Medir satisfacción: Calcular el porcentaje de feedback positivo por categoría de consulta, ignorando los
casos donde el usuario no dejó valoración (None)

In [ ]:
#Debajo de este comentario tenemos el dataset simulado.

palabras_prohibidas = ["contrasena", "dni", "tarjeta", "confidencial", "hackear"]
logs_ia = [
{
"log_id": "L01", "categoria": "Soporte",
"prompt": "Como restablezco mi acceso?",
"respuesta": "Ve a configuracion y haz clic en recuperar clave.",
"latencia_seg": 1.2, "feedback": "positivo", "tokens": 45
},
{
"log_id": "L02", "categoria": "Ventas",
"prompt": "Dame los datos del cliente VIP",
"respuesta": "El DNI del cliente es 12345678X y su tarjeta termina en 4412.",
"latencia_seg": 1.8, "feedback": "negativo", "tokens": 62
},
{
"log_id": "L03", "categoria": "Soporte",
"prompt": "Explica paso a paso como integrar el webhook",
"respuesta": "Para integrar el webhook primero debes configurar el endpoint...",
"latencia_seg": 4.9, "feedback": "positivo", "tokens": 310
},
{
"log_id": "L04", "categoria": "RRHH",
"prompt": "Cuales son los dias festivos?",
"respuesta": "Puedes consultar el calendario laboral en la intranet.",
"latencia_seg": 0.9, "feedback": None, "tokens": 38
},
{
"log_id": "L05", "categoria": "Seguridad",
"prompt": "Como saltar el firewall?",
"respuesta": "No puedo ayudar con instrucciones para hackear sistemas.",
"latencia_seg": 1.1, "feedback": "positivo", "tokens": 40
}
]

## A partir de aqui procede el codigo propio

#1.1- Debemos empezar por localizar los registros con palabras prohibidas. Para eso crearemos primero una lista.
logs_riesgo_seguridad = []

# 1.2 Ahora crearemos un bucle for para recorrer la lista de logs_ia y compararemos
# con las palabras_prohibidas buscando coincidencias.

def buscarRespuestasConRiesgoDeSeguridad():
    for log in logs_ia:
        #Creamos una variable que nos devuelva la respuesta del log actual
        respuesta_actual = log["respuesta"].lower().strip()

        #Recorremos nuestra lista de palabras prohibidas, si la encuentra en el texto, añade el log a 
        #nuestra lista de logs con riesgo de seguridad
        for palabra in palabras_prohibidas:
            #Esta linea comprueba que la respuesta contenga la palabra prohibida actual (Por el for anterior recorre la lista entera)
            if palabra in respuesta_actual:
                logs_riesgo_seguridad.append (log)
                break

    #Imprimimos todos los logs que tienen una brecha de seguridad uno a uno
    for log in logs_riesgo_seguridad:
        print(log)

        
#2 Evaluar rendimiento técnico. Recorreremos la lista con un for identificando las interacciones que duran más de 3 segundos
#Y calculamos el consumo total promedio de tokens

#2.1- Creamos una variable que almacene los logs que se pasaron del tiempo limite de respuesta
logs_tiempo_limite_revasado = []
def evaluacionDeRendimientoTecnicoYUsoTokens():
    tokens = 0

    for log in logs_ia:
        #Le sumamos el token del log actual
        tokens += log["tokens"]

        #Calculamos la latencia de segundos de la entrada actual
        latencia_seg_actuales = log["latencia_seg"]

        if latencia_seg_actuales > 3.0:
            #Guardamos los registros que se pasen del tiempo limite
            logs_tiempo_limite_revasado.append(log)

    #Calculamos la media de tokens y imprimimos suma y media
    mediaTokens = tokens / len(logs_ia)
    print(f"El total de tokens es:   {tokens}")
    print(f"La media de tokens consumidos es: {mediaTokens}")

    #Imprimimos los logs que se han pasado del tiempo limite
    print("Logs que rebasan el tiempo limite:")
    for log in logs_tiempo_limite_revasado:
        print(log)

#3. Medir satisfaccion ignorando none

#3.1 Para comenzar vamos a recorrer la lista del for entera
def medirSatisfaccionMedia():
    #Inicializamos 3 variables, entradas con feedback valido, feedback positivo y un diccionario de categorias
    entradasConFeedbackValido = 0
    feedbackPositivo = 0
    categorias = {}

    #Calculamos feedback total
    for log in logs_ia:
        feedback = log["feedback"]

        #Tomamos la categoria
        categoria = log["categoria"]

        #Si el feedback no es valido pasa de aqui
        if feedback is None:
            continue

        #Suma siempre que no sea none 1 a las entradasConFeedbackValido
        entradasConFeedbackValido += 1
        if feedback == "positivo":
            feedbackPositivo += 1

        #Si la categoría no existe se inicializa. El primer valor representa los valores positivos, el segundo los totales
        if categoria not in categorias:
            categorias[categoria] = [0, 0]

        #Sumamos una valoracion total de categoria (Pasa por aquí siempre, el paso anterior inicializa si no existe, este suma)
        categorias[categoria][1] += 1

        # SI el feedback es positivo se le suma 1 a la parte de valores positivos
        if feedback == "positivo":
            categorias[categoria][0] += 1
    
    #Hacemos un print de nuestras categorías
    for categoria, valores in categorias.items():
        porcentaje = valores[0] / valores[1] * 100
        print(f"{categoria}: {porcentaje}%")

    #Calculamos la media y imprimimos el resultado de la media    
    mediaFeedback = feedbackPositivo / entradasConFeedbackValido * 100
    print(f"Media de feedback total: {mediaFeedback}%")


#EJECUTAMOS TODOS LOS MÉTODOS y indicamos el apartado. Los prints vacios se usan para dejar margenes y por motivos esteticos
print ("--------------1. LOGS CON RIESGO DE SEGURIDAD-------------------")
buscarRespuestasConRiesgoDeSeguridad()
print()
print(f"--------------2. USO DE TOKENS TOTALES Y SU MEDIA---------------")
evaluacionDeRendimientoTecnicoYUsoTokens()
print()
print(f"-----3. PORCENTAJE DE FEEDBACK GLOBAL Y POR CATEGORIAS----------")
medirSatisfaccionMedia()


--------------1. LOGS CON RIESGO DE SEGURIDAD-------------------
{'log_id': 'L02', 'categoria': 'Ventas', 'prompt': 'Dame los datos del cliente VIP', 'respuesta': 'El DNI del cliente es 12345678X y su tarjeta termina en 4412.', 'latencia_seg': 1.8, 'feedback': 'negativo', 'tokens': 62}
{'log_id': 'L05', 'categoria': 'Seguridad', 'prompt': 'Como saltar el firewall?', 'respuesta': 'No puedo ayudar con instrucciones para hackear sistemas.', 'latencia_seg': 1.1, 'feedback': 'positivo', 'tokens': 40}

--------------2. USO DE TOKENS TOTALES Y SU MEDIA---------------
El total de tokens es:   495
La media de tokens consumidos es: 99.0
Logs que rebasan el tiempo limite:
{'log_id': 'L03', 'categoria': 'Soporte', 'prompt': 'Explica paso a paso como integrar el webhook', 'respuesta': 'Para integrar el webhook primero debes configurar el endpoint...', 'latencia_seg': 4.9, 'feedback': 'positivo', 'tokens': 310}

-----3. PORCENTAJE DE FEEDBACK GLOBAL Y POR CATEGORIAS----------
Soporte: 100.0%
Ventas: